# Altair playground
A testing area for Altair plots

In [2]:
import altair as alt
from harpy.report.utilities import palette, trunc_digits
from harpy.report.theme import sv_colors
from altair.datasets import data
import pandas as pd
import altair as alt
import pandas as pd
import numpy as np
from altair.datasets import data

In [15]:
cars = data.cars.url
labels = ['Europe', 'Japan', 'USA']
input_dropdown = alt.binding_select(options=labels, name='Region ')
selection = alt.selection_point(fields=['Origin'], value = labels[0], bind=input_dropdown)

(
    alt.Chart(cars)
    .mark_point()
    .encode(
        x='Horsepower:Q',
        y='Miles_per_Gallon:Q',
        color=alt.Color('Origin:N').scale(domain = labels).legend(None),
    )
    .add_params(selection)
    .transform_filter(selection)
    .properties(title = "HOO HAA")
)


alt.Chart(...)

In [3]:
inversions = pd.DataFrame({
    'chromosome': ['chr1', 'chr1', 'chr2', 'chr2', 'chr3', 'chr1', 'chr3'],
    'start': [1000000, 5000000, 2000000, 8000000, 3000000, 7000000, 6000000],
    'end': [2000000, 6500000, 3500000, 9500000, 5000000, 8500000, 7500000],
    'variant': ['INV','INV','INV','INV','INV','INV','INV']

})

# Sample data: deletions
deletions = pd.DataFrame({
    'chromosome': ['chr1', 'chr2', 'chr2', 'chr3', 'chr3', 'chr1', 'chr2'],
    'start': [4000, 1000000, 6000000, 1000000, 6000000, 8000000, 10000000],
    'end': [6000, 1800000, 7000000, 2000000, 7000000, 9000000, 11000000],
    'variant': ['DEL','DEL','DEL','DEL','DEL','DEL','DEL'],
})


sv = pd.concat([inversions,deletions])
sv.head()


,chromosome,start,end,variant
0,chr1,1000000,2000000,INV
1,chr1,5000000,6500000,INV
2,chr2,2000000,3500000,INV
3,chr2,8000000,9500000,INV
4,chr3,3000000,5000000,INV


In [4]:
# Define chromosome lengths (example values - adjust to your genome)
chr_lengths = {
    'chr1': 10000000,
    'chr2': 12000000,
    'chr3': 8000000
}
chrs = pd.DataFrame(chr_lengths.items(), columns = ["chromosome", "length"])
chrs.head()

,chromosome,length
0,chr1,10000000
1,chr2,12000000
2,chr3,8000000


In [5]:
var_df = sv.merge(chrs, "left", on = "chromosome")
var_df.head()

,chromosome,start,end,variant,length
0,chr1,1000000,2000000,INV,10000000
1,chr1,5000000,6500000,INV,10000000
2,chr2,2000000,3500000,INV,12000000
3,chr2,8000000,9500000,INV,12000000
4,chr3,3000000,5000000,INV,8000000


In [80]:
import altair as alt
labels = var_df['chromosome'].unique()
input_dropdown = alt.binding_select(options=labels, name='Chromosome ')
selection = alt.selection_point(fields=['chromosome'], value = labels[0], bind=input_dropdown)

(
    alt.Chart(var_df)
    .mark_rect(strokeWidth=2, cornerRadius=8)
    .encode(
        x=alt.X('start:Q'), #.scale(domain = [0, {'expr': 'data("source_0")[0].length'}]),
        x2='end:Q',
        y='variant:N',
        color='variant:N',
        tooltip=['variant:N', 'chromosome:N', 'start:Q', 'end:Q']
    )
    .add_params(selection)
    .transform_filter(selection)
    .properties(title = "HOO HAA")
)

alt.Chart(...)

In [ ]:
import altair as alt

labels = var_df['chromosome'].unique()
input_dropdown = alt.binding_select(options=labels, name='Chromosome ')
selection = alt.selection_point(fields=['chromosome'], value=labels[0], bind=input_dropdown)
length_param = alt.param(
    name='chr_length',
    expr='data("data_0")[0].length'
)
(
    alt.Chart(var_df)
    .mark_rect(strokeWidth=2, cornerRadius=8)
    .encode(
        x=alt.X('start:Q', scale=alt.Scale(domain=[0, length_param])),
        x2='end:Q',
        y='variant:N',
        color='variant:N',
        tooltip=['variant:N', 'chromosome:N', 'start:Q', 'end:Q']
    )
    .add_params(selection)
    .transform_filter(selection)
    .add_params(length_param)
    .properties(title="HOO HAA")
)

alt.Chart(...)

In [ ]:
def by_chromosome_plot(variants: pd.DataFrame, title:str = ""):
    labels = variants['chromosome'].unique()
    input_dropdown = alt.binding_select(options=labels, name='Chromosome: ')
    selection = alt.selection_point("chrom_choice", fields=['chromosome'], value=labels[0], bind=input_dropdown)
    length_param = alt.param(expr='data("data_0")[0].length')
    highlight = alt.selection_point(name="highlight", on="pointerover", empty=False)

    stroke_width = (
        alt.when(highlight).then(alt.value("lightgrey"))
        .otherwise(alt.value("transparent"))
    )

    return (
        alt.Chart(variants)
        .mark_rect(strokeWidth=2, cornerRadius=8)
        .encode(
            x=alt.X('start:Q')
                .scale(alt.Scale(domain=[0, length_param]))
                .axis(alt.Axis(title='Position (Mb)', labelExpr='datum.value / 1000000')),
            x2='end:Q',
            y='variant:N',
            color='variant:N',
            tooltip=['variant:N', 'chromosome:N', 'start:Q', 'end:Q'],
            stroke=stroke_width
        )
        .transform_filter(selection)
        .add_params(selection, length_param, highlight)
        .properties(title= title)
    )

by_chromosome_plot(var_df)

alt.Chart(...)